In [1]:
# Data handling and mathematics
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Feature preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Feature selection
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression

# Principal Component Analysis
from sklearn.decomposition import PCA

# Time-series validation and feature evaluation
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# File and folder paths
from pathlib import Path

# Plot settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("Feature Engineering and PCA libraries imported successfully.")

Feature Engineering and PCA libraries imported successfully.


In [2]:
# Load the saved Feature Engineering dataset
pca_ready = pd.read_csv(
    "../data/energy_1hour_fe_ready.csv",
    parse_dates=["date"]
).sort_values("date").reset_index(drop=True)

# Separate timestamps, target, and predictors
timestamps = pca_ready["date"].copy()
y = pca_ready["Total_Energy"].copy()
X = pca_ready.drop(columns=["date", "Total_Energy"]).copy()

# Restore the chronological 80%–20% split
split_index = int(len(X) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()
y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Training inputs:", X_train.shape)
print("Testing inputs:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training inputs: (2631, 41)
Testing inputs: (658, 41)
Training target: (2631,)
Testing target: (658,)


In [3]:
# Select original sensor and weather measurements for PCA
pca_columns = (
    [f"T{i}" for i in range(1, 10)]
    + [f"RH_{i}" for i in range(1, 10)]
    + [
        "T_out", "Press_mm_hg", "RH_out",
        "Windspeed", "Visibility", "Tdewpoint"
    ]
)

X_train_pca_input = X_train[pca_columns].copy()
X_test_pca_input = X_test[pca_columns].copy()

print("Training PCA inputs:", X_train_pca_input.shape)
print("Testing PCA inputs:", X_test_pca_input.shape)

Training PCA inputs: (2631, 24)
Testing PCA inputs: (658, 24)


In [4]:
# Learn scaling parameters from training data only
pca_scaler = StandardScaler()

Z_train = pd.DataFrame(
    pca_scaler.fit_transform(X_train_pca_input),
    columns=pca_columns,
    index=X_train_pca_input.index
)

# Apply the same training parameters to test data
Z_test = pd.DataFrame(
    pca_scaler.transform(X_test_pca_input),
    columns=pca_columns,
    index=X_test_pca_input.index
)

# Verify training feature means and standard deviations
scaling_check = pd.DataFrame({
    "Training_mean": Z_train.mean(),
    "Training_std": Z_train.std(ddof=0)
})

display(scaling_check.round(6))

,Training_mean,Training_std
T1,0.0,1.0
T2,-0.0,1.0
T3,0.0,1.0
T4,0.0,1.0
T5,0.0,1.0
T6,-0.0,1.0
T7,-0.0,1.0
T8,-0.0,1.0
T9,0.0,1.0
RH_1,0.0,1.0


In [5]:
# Compute covariance using standardized training data only
cov_matrix = Z_train.cov()

print("Covariance matrix shape:", cov_matrix.shape)

# Inspect a small section first
display(
    cov_matrix.loc[
        ["T1", "T2", "T3", "T_out"],
        ["T1", "T2", "T3", "T_out"]
    ].round(4)
)

Covariance matrix shape: (24, 24)


,T1,T2,T3,T_out
T1,1.0004,0.7622,0.8103,0.4366
T2,0.7622,1.0004,0.5569,0.6006
T3,0.8103,0.5569,1.0004,0.5004
T_out,0.4366,0.6006,0.5004,1.0004


In [6]:
# Find eigenvalues and eigenvectors of the symmetric covariance matrix
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix.to_numpy())

# Sort from greatest to smallest variance
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

# Calculate individual and cumulative explained variance
variance_ratio = eigenvalues / eigenvalues.sum()

variance_table = pd.DataFrame({
    "Component": [f"PC{i}" for i in range(1, len(eigenvalues) + 1)],
    "Eigenvalue": eigenvalues,
    "Explained_variance_%": variance_ratio * 100,
    "Cumulative_variance_%": np.cumsum(variance_ratio) * 100
})

display(variance_table.round(4))

,Component,Eigenvalue,Explained_variance_%,Cumulative_variance_%
0,PC1,8.3567,34.8065,34.8065
1,PC2,6.7740,28.2144,63.0210
2,PC3,2.1112,8.7935,71.8145
3,PC4,1.1978,4.9889,76.8034
4,PC5,1.0470,4.3610,81.1643
5,PC6,0.9982,4.1574,85.3217
6,PC7,0.7010,2.9196,88.2414
7,PC8,0.5457,2.2730,90.5144
8,PC9,0.4895,2.0389,92.5534
9,PC10,0.3366,1.4020,93.9554


In [7]:
# Inspect the feature weights defining PC1
pc1_weights = pd.DataFrame({
    "Feature": pca_columns,
    "PC1_weight": eigenvectors[:, 0]
})

pc1_weights["Absolute_weight"] = pc1_weights["PC1_weight"].abs()

display(
    pc1_weights
    .sort_values("Absolute_weight", ascending=False)
    .reset_index(drop=True)
    .round(4)
)

,Feature,PC1_weight,Absolute_weight
0,RH_6,0.3062,0.3062
1,T7,-0.2932,0.2932
2,T9,-0.2786,0.2786
3,T8,-0.2699,0.2699
4,RH_8,0.2628,0.2628
5,RH_3,0.2588,0.2588
6,T4,-0.2523,0.2523
7,RH_7,0.2388,0.2388
8,RH_4,0.2378,0.2378
9,RH_9,0.2303,0.2303


In [8]:
# Load the saved Feature Engineering dataset
six_fe_ready = pd.read_csv(
    "../data/energy_1hour_fe_ready.csv",
    parse_dates=["date"]
).sort_values("date").reset_index(drop=True)

# Separate timestamps, target, and predictors
timestamps = six_fe_ready["date"].copy()
y = six_fe_ready["Total_Energy"].copy()
X = six_fe_ready.drop(columns=["date", "Total_Energy"]).copy()

# Restore the chronological 80%–20% split
split_index = int(len(X) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()
y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

# Select your six features for PCA
floor_weather_columns = [
    "Floor_1_Temp",
    "Floor_2_Temp",
    "Floor_1_Humidity",
    "Floor_2_Humidity",
    "T_out",
    "RH_out"
]

X_train_floor_input = X_train[floor_weather_columns].copy()
X_test_floor_input = X_test[floor_weather_columns].copy()

print("Training PCA inputs:", X_train_floor_input.shape)
print("Testing PCA inputs:", X_test_floor_input.shape)

display(X_train_floor_input.head())

Training PCA inputs: (2631, 6)
Testing PCA inputs: (658, 6)


,Floor_1_Temp,Floor_2_Temp,Floor_1_Humidity,Floor_2_Humidity,T_out,RH_out
0,19.626667,17.688778,45.342361,47.282528,6.308333,92.000000
1,19.645556,17.665028,45.060417,47.395694,5.941667,91.583333
2,20.157778,17.802778,48.698194,48.645583,6.000000,89.750000
3,20.475370,18.020389,47.118287,52.298028,6.000000,87.583333
4,20.757593,18.351944,45.631296,50.985000,5.833333,87.416667


In [9]:
# Learn scaling parameters from the six training features only
six_scaler = StandardScaler()

Z_train_six = pd.DataFrame(
    six_scaler.fit_transform(X_train_floor_input),
    columns=floor_weather_columns,
    index=X_train_floor_input.index
)

# Apply the training parameters to the test features
Z_test_six = pd.DataFrame(
    six_scaler.transform(X_test_floor_input),
    columns=floor_weather_columns,
    index=X_test_floor_input.index
)

# Check training standardization
display(
    pd.DataFrame({
        "Training_mean": Z_train_six.mean(),
        "Training_std": Z_train_six.std(ddof=0)
    }).round(6)
)

,Training_mean,Training_std
Floor_1_Temp,0.0,1.0
Floor_2_Temp,-0.0,1.0
Floor_1_Humidity,0.0,1.0
Floor_2_Humidity,-0.0,1.0
T_out,-0.0,1.0
RH_out,-0.0,1.0


In [10]:
# Fit PCA on the six standardized training features only
six_pca = PCA(svd_solver="full")
six_pca.fit(Z_train_six)

component_names = [
    f"PC{i}" for i in range(1, six_pca.n_components_ + 1)
]

# Feature weights defining each component
six_pca_weights = pd.DataFrame(
    six_pca.components_.T,
    index=floor_weather_columns,
    columns=component_names
)

print("PCA weights:")
display(six_pca_weights.round(4))

# Individual and cumulative explained variance
six_variance_table = pd.DataFrame({
    "Component": component_names,
    "Eigenvalue": six_pca.explained_variance_,
    "Explained_variance_%": six_pca.explained_variance_ratio_ * 100,
    "Cumulative_variance_%": (
        np.cumsum(six_pca.explained_variance_ratio_) * 100
    )
})

print("\nExplained variance:")
display(six_variance_table.round(4))

PCA weights:


,PC1,PC2,PC3,PC4,PC5,PC6
Floor_1_Temp,0.4619,0.3493,0.3590,-0.3662,-0.3755,-0.5106
Floor_2_Temp,0.5133,0.1155,0.5155,0.1156,0.4478,0.4935
Floor_1_Humidity,-0.2638,0.5973,-0.0633,0.0717,0.6387,-0.3957
Floor_2_Humidity,-0.3170,0.5591,-0.0378,-0.4407,-0.2538,0.5718
T_out,0.3696,0.4312,-0.4363,0.6213,-0.3022,0.0992
RH_out,-0.4654,0.0971,0.6400,0.5169,-0.3080,-0.0488



Explained variance:


,Component,Eigenvalue,Explained_variance_%,Cumulative_variance_%
0,PC1,2.6037,43.3793,43.3793
1,PC2,2.0351,33.9052,77.2845
2,PC3,0.8212,13.6812,90.9657
3,PC4,0.2426,4.0413,95.0070
4,PC5,0.1658,2.7628,97.7698
5,PC6,0.1339,2.2302,100.0000


In [11]:
# Version 1: 24 original measurements -> 11 components
room_columns = (
    [f"T{i}" for i in range(1, 10)]
    + [f"RH_{i}" for i in range(1, 10)]
    + [
        "T_out", "Press_mm_hg", "RH_out",
        "Windspeed", "Visibility", "Tdewpoint"
    ]
)

room_scaler = StandardScaler()

Z_train_room = room_scaler.fit_transform(X_train[room_columns])
Z_test_room = room_scaler.transform(X_test[room_columns])

pca_11 = PCA(n_components=11, svd_solver="full")
pc11_names = [f"PC{i}" for i in range(1, 12)]

X_train_pc11 = pd.DataFrame(
    pca_11.fit_transform(Z_train_room),
    index=X_train.index,
    columns=pc11_names
)

X_test_pc11 = pd.DataFrame(
    pca_11.transform(Z_test_room),
    index=X_test.index,
    columns=pc11_names
)

# Version 2: six floor/weather measurements -> six components
floor_weather_columns = [
    "Floor_1_Temp",
    "Floor_2_Temp",
    "Floor_1_Humidity",
    "Floor_2_Humidity",
    "T_out",
    "RH_out"
]

six_scaler = StandardScaler()

Z_train_six = six_scaler.fit_transform(X_train[floor_weather_columns])
Z_test_six = six_scaler.transform(X_test[floor_weather_columns])

six_pca = PCA(n_components=6, svd_solver="full")
pc6_names = [f"PC{i}" for i in range(1, 7)]

X_train_pc6 = pd.DataFrame(
    six_pca.fit_transform(Z_train_six),
    index=X_train.index,
    columns=pc6_names
)

X_test_pc6 = pd.DataFrame(
    six_pca.transform(Z_test_six),
    index=X_test.index,
    columns=pc6_names
)

# Check the two representations
print("11-PC training/testing:", X_train_pc11.shape, X_test_pc11.shape)
print("6-PC training/testing:", X_train_pc6.shape, X_test_pc6.shape)

print(
    "\n11-PC retained training variance:",
    f"{pca_11.explained_variance_ratio_.sum():.4%}"
)
print(
    "6-PC retained training variance:",
    f"{six_pca.explained_variance_ratio_.sum():.4%}"
)

11-PC training/testing: (2631, 11) (658, 11)
6-PC training/testing: (2631, 6) (658, 6)

11-PC retained training variance: 95.2168%
6-PC retained training variance: 100.0000%


In [12]:
# Prepare Model B with the agreed nine calendar features
calendar_columns = [
    "hour_sin",
    "hour_cos",
    "is_weekend",
    "is_afternoon_peak",
    "month_2",
    "month_3",
    "month_4",
    "month_5",
    "Weekend_Afternoon"
]

assert X_train_pc6.index.equals(X_train.index)
assert X_test_pc6.index.equals(X_test.index)

X_train_modelB = pd.concat(
    [X_train_pc6, X_train[calendar_columns]], axis=1
)
X_test_modelB = pd.concat(
    [X_test_pc6, X_test[calendar_columns]], axis=1
)

X_train_modelB["is_weekend"] = X_train_modelB["is_weekend"].astype(int)
X_test_modelB["is_weekend"] = X_test_modelB["is_weekend"].astype(int)


# Attach timestamps, target, and split labels
def prepare_model_csv(train_inputs, test_inputs):
    assert train_inputs.index.equals(y_train.index)
    assert test_inputs.index.equals(y_test.index)

    train_data = train_inputs.copy()
    train_data.insert(0, "date", timestamps.loc[train_inputs.index])
    train_data["Total_Energy"] = y_train
    train_data["split"] = "train"

    test_data = test_inputs.copy()
    test_data.insert(0, "date", timestamps.loc[test_inputs.index])
    test_data["Total_Energy"] = y_test
    test_data["split"] = "test"

    combined = pd.concat([train_data, test_data]).sort_values("date")
    assert not combined.isna().any().any(), "Unexpected missing values."
    return combined


# Prepare both datasets
modelA_ready = prepare_model_csv(X_train_pc11, X_test_pc11)
modelB_ready = prepare_model_csv(X_train_modelB, X_test_modelB)

# Save CSV files
output_folder = Path("../data")
output_folder.mkdir(parents=True, exist_ok=True)

modelA_path = output_folder / "energy_model_A_pca11.csv"
modelB_path = output_folder / "energy_model_B_pca6_calendar.csv"

modelA_ready.to_csv(modelA_path, index=False)
modelB_ready.to_csv(modelB_path, index=False)

print("Model A saved:", modelA_path)
print("Model A CSV shape:", modelA_ready.shape)

print("\nModel B saved:", modelB_path)
print("Model B CSV shape:", modelB_ready.shape)

Model A saved: ..\data\energy_model_A_pca11.csv
Model A CSV shape: (3289, 14)

Model B saved: ..\data\energy_model_B_pca6_calendar.csv
Model B CSV shape: (3289, 18)


In [13]:
summary = """DS605 HOUSEHOLD ENERGY PREDICTION
PCA PROGRESS SUMMARY
Date: 5 November 2026

1. STARTING DATA AND PREDICTION TASK

Input file: data/energy_1hour_fe_ready.csv
Dataset shape: 3,289 rows and 43 columns.
Contents: 41 predictors, date, and Total_Energy.

Goal: estimate current-hour Total_Energy using current-hour measurements.
Target unit: Wh per hour.

2. CHRONOLOGICAL TRAIN-TEST SPLIT

Training: 2,631 hours (earliest 80%).
Period: 2016-01-11 17:00 to 2016-04-30 07:00.

Testing: 658 hours (latest 20%).
Period: 2016-04-30 08:00 to 2016-05-27 17:00.

Scaling and PCA were fitted on training measurements only.
Test measurements were transformed using the same fitted objects.
Total_Energy was excluded from PCA.

3. PCA MATHEMATICS STUDIED

Standardization:
z = (measurement - training mean) / training standard deviation.

Training standardized features had means approximately 0 and standard
deviations approximately 1. Test statistics need not equal these values.

Covariance:
S = Z_centered.T @ Z_centered / (n - 1)

Covariance describes how measurements vary together.
Positive covariance indicates shared movement; it does not prove causation.

Eigenvalue equation:
S w = lambda w

Eigenvectors provide component directions and feature weights.
Eigenvalues measure variance captured by each component.

Component score:
PC_k = sum(weight_jk * centered_standardized_feature_j)

Explained variance ratio:
EVR_k = eigenvalue_k / sum(all eigenvalues)

Cumulative explained variance adds the ratios of successive components.

PCA captures variation in inputs, not energy-prediction accuracy.
95% variance does not mean 95% of observations or energy consumption.
PCA does not create clusters.

4. MODEL A: ORIGINAL 24 MEASUREMENTS

PCA inputs:
- T1 to T9
- RH_1 to RH_9
- T_out
- Press_mm_hg
- RH_out
- Windspeed
- Visibility
- Tdewpoint

Standardization was followed by covariance analysis and eigenvalue/
eigenvector calculation. PCA feature weights were inspected.

Selected cumulative explained variance:
2 components: 63.0210%
8 components: 90.5144%
10 components: 93.9554%
11 components: 95.2168%
24 components: 100%

Decision: retain 11 components.
Variance omitted: 4.7832%.

Model A inputs: PC1 to PC11 only.
Training shape: (2631, 11).
Testing shape: (658, 11).

Final preprocessing objects:
room_scaler and pca_11.

5. MODEL B: SIX FLOOR/OUTDOOR MEASUREMENTS

Selected inputs:
- Floor_1_Temp
- Floor_2_Temp
- Floor_1_Humidity
- Floor_2_Humidity
- T_out
- RH_out

Existing floor averages were reused.
A separate scaler and PCA were fitted for this feature set.

Press_mm_hg, Windspeed, Visibility, and Tdewpoint were excluded from
this version by project choice. Low correlation alone does not establish
that a feature has no predictive value.

Component | Individual variance % | Cumulative variance %
PC1       | 43.3793               | 43.3793
PC2       | 33.9052               | 77.2845
PC3       | 13.6812               | 90.9657
PC4       |  4.0413               | 95.0070
PC5       |  2.7628               | 97.7698
PC6       |  2.2302               | 100.0000

Four components would retain approximately 95% variance.
We chose all six components, retaining 100% of the six-feature variance.
Keeping six components from six measurements rotates the representation;
it does not reduce dimensionality or restore excluded room-level details.

PC1 contrasts higher temperatures with lower relative humidities.
PC2 mainly represents shared indoor humidity variation, with temperature
contributions.

PCA weights describe component directions, not importance for predicting
energy. A component's signs may all reverse without changing its meaning.

Final preprocessing objects:
six_scaler and six_pca.

6. MODEL B CALENDAR FEATURES

Nine calendar features were added directly AFTER PCA:
1. hour_sin
2. hour_cos
3. is_weekend
4. is_afternoon_peak
5. month_2
6. month_3
7. month_4
8. month_5
9. Weekend_Afternoon

hour_sin = sin(2*pi*hour/24)
hour_cos = cos(2*pi*hour/24)

is_afternoon_peak = 1 for 15 <= hour < 19, otherwise 0.
Weekend_Afternoon = is_weekend * is_afternoon_peak.

January is represented by all four month indicators being zero.
The six weekday dummy features were removed by user choice.
Floor-temperature changes and Both_Floors_Warming were excluded.

Calendar features remain directly interpretable inputs.
They do not require PCA. Their numerical ranges are not the main reason
for keeping them outside PCA.

Model B final inputs:
6 PCA scores + 9 calendar features = 15 predictors.

Training shape: (2631, 15).
Testing shape: (658, 15).

7. FINAL MODEL DESIGNS

Model A: 11 PCA scores only.
Model B: 6 PCA scores plus nine calendar features.
Both predict Total_Energy using the same training/test hours.

No separate calendar-only Model C is planned.
These designs differ in sensor selection and calendar inputs, so their
comparison will not isolate the effect of PCA alone.

8. SAVED CSV FILES

Model A:
data/energy_model_A_pca11.csv
Confirmed shape: (3289, 14).
Contains date, 11 PCA scores, Total_Energy, and split.

Model B:
data/energy_model_B_pca6_calendar.csv
Confirmed shape: (3289, 18).
Contains date, six PCA scores, nine calendar features, Total_Energy,
and split.

Use split to preserve the original training/testing assignment.
Exclude date, Total_Energy, and split from model predictors.
Use Total_Energy as the target.

9. FRONTEND PLAN

Model B can accept seven form fields:
- Date/time
- Floor 1 average temperature
- Floor 2 average temperature
- Floor 1 average relative humidity
- Floor 2 average relative humidity
- Outdoor temperature
- Outdoor relative humidity

The backend uses the saved scaler and PCA to create six component scores.
Date/time generates nine calendar features.
These together create 15 prediction-model inputs.

10. CURRENT STATUS AND NEXT STEPS

Completed:
- Training/test split
- Standardization
- Covariance and eigenvalue/eigenvector study
- Explained-variance and component-weight interpretation
- Both PCA transformations
- Calendar-input selection
- Model A and Model B CSV exports

Not yet completed:
- Saving fitted scalers and PCA objects for frontend reuse
- Prediction-model training and evaluation
- Frontend implementation

Important modeling issue:
May occurs only in testing. month_5 is always zero during training,
so a May-specific effect cannot be learned. Review month representation
before final modeling. The data does not cover a full annual cycle.

For time-series cross-validation, refit scaling and PCA within each
training fold using the original measurements. Do not reuse PCA scores
fitted on the entire training period across validation folds.

Next stage: machine learning model training and evaluation,
one step at a time.
"""

summary_path = Path("../txt_file/PCA_Progress_Summary.txt")
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(summary, encoding="utf-8")

print("Summary saved:", summary_path)

Summary saved: ..\txt_file\PCA_Progress_Summary.txt
